# Detección Automática de Uso de Cascos de Seguridad en Entornos de Construcción

## Proyecto de Inteligencia Artificial

Este notebook implementa un sistema de detección de objetos basado en YOLOv8 para identificar trabajadores con casco y sin casco en imágenes de entornos de construcción.

## Modos de uso

### Opción A — Entrenar desde cero
Esta opción descarga el dataset, construye un subconjunto de 1000 imágenes y entrena el modelo YOLOv8n.

### Opción B — Usar un modelo ya entrenado
Esta opción carga directamente el archivo `helmet.pt` y realiza predicciones sobre imágenes nuevas, sin necesidad de repetir el entrenamiento.

## Clases utilizadas
- `helmet` → trabajador con casco
- `head` → trabajador sin casco
- `person` → clase adicional presente en el dataset original

# 1. Instalación de dependencias

In [ ]:
!pip install -q roboflow ultralytics

# 2. Importación de librerías

In [ ]:
from roboflow import Roboflow
from ultralytics import YOLO
import random
import shutil
from pathlib import Path
import os
from IPython.display import Image, display

# 3. Opción A — Entrenamiento desde cero

Ejecuta esta sección solo si deseas volver a entrenar el modelo.
Si ya cuentas con el archivo `helmet.pt`, puedes saltar directamente a la sección de uso del modelo entrenado.

# 3. Descarga del dataset desde Roboflow

En esta sección se descarga la versión 2 del dataset **Hard Hat Workers** en formato compatible con YOLOv8.

In [ ]:
rf = Roboflow(api_key="---------------")

project = rf.workspace("joseph-nelson").project("hard-hat-workers")
version = project.version(2)
dataset = version.download("yolov8")

print("Dataset descargado en:", dataset.location)

## 3.2 Creación del subconjunto de 1000 imágenes

Debido a limitaciones computacionales del entorno de ejecución, se construye un subconjunto representativo de 1000 imágenes a partir del conjunto original de entrenamiento.

## División utilizada
- 800 imágenes para entrenamiento
- 200 imágenes para validación

In [ ]:
random.seed(42)

base = Path(dataset.location)
train_img_dir = base / "train" / "images"
train_lbl_dir = base / "train" / "labels"

small_base = Path("/content/Hard-Hat-Workers-1000")
for split in ["train", "valid"]:
    (small_base / split / "images").mkdir(parents=True, exist_ok=True)
    (small_base / split / "labels").mkdir(parents=True, exist_ok=True)

all_images = [p for p in train_img_dir.iterdir() if p.is_file()]
print("Imágenes disponibles en train:", len(all_images))

selected = random.sample(all_images, 1000)

train_selected = selected[:800]
valid_selected = selected[800:]

def copy_pairs(img_list, out_img_dir, out_lbl_dir):
    copied = 0
    missing = 0
    for img_path in img_list:
        label_path = train_lbl_dir / f"{img_path.stem}.txt"
        if label_path.exists():
            shutil.copy2(img_path, out_img_dir / img_path.name)
            shutil.copy2(label_path, out_lbl_dir / label_path.name)
            copied += 1
        else:
            missing += 1
    return copied, missing

train_copied, train_missing = copy_pairs(
    train_selected,
    small_base / "train" / "images",
    small_base / "train" / "labels"
)

valid_copied, valid_missing = copy_pairs(
    valid_selected,
    small_base / "valid" / "images",
    small_base / "valid" / "labels"
)

print("Train copiadas:", train_copied, "| faltantes:", train_missing)
print("Valid copiadas:", valid_copied, "| faltantes:", valid_missing)
print("Total final:", train_copied + valid_copied)

## 3.3 Configuración del dataset

Se crea el archivo `data.yaml`, que define:
- la ruta de entrenamiento
- la ruta de validación
- el número de clases
- los nombres de las clases





In [ ]:
%%writefile /content/Hard-Hat-Workers-1000/data.yaml
train: /content/Hard-Hat-Workers-1000/train/images
val: /content/Hard-Hat-Workers-1000/valid/images

nc: 3
names: ['head', 'helmet', 'person']

## 3.4 Entrenamiento del modelo YOLOv8n

Se utiliza el modelo base `yolov8n.pt` por su ligereza y eficiencia computacional.  
La configuración utilizada fue:

- Épocas: 10
- Tamaño de imagen: 416
- Batch size: 8

In [ ]:
model = YOLO("yolov8n.pt")

model.train(
    data="/content/Hard-Hat-Workers-1000/data.yaml",
    epochs=10,
    imgsz=416,
    batch=8
)

## 3.5 Exportación del modelo final

Al finalizar el entrenamiento, se copia el mejor modelo entrenado a un archivo llamado `helmet.pt`.

In [ ]:
runs_dir = Path("/content/runs/detect")
latest_train = sorted(runs_dir.glob("train*"))[-1]
best_model_path = latest_train / "weights" / "best.pt"

print("Mejor modelo encontrado en:", best_model_path)

In [ ]:
!cp "{best_model_path}" /content/helmet.pt
print("Modelo exportado como /content/helmet.pt")

## 3.6 Validación final del modelo

Se realiza una validación final del modelo entrenado para obtener métricas de desempeño:
- Precision
- Recall
- mAP@50
- mAP@50-95

In [ ]:
model = YOLO("/content/helmet.pt")
metrics = model.val()
print(metrics)

## 3.7 Descarga de archivos importantes

In [ ]:
from google.colab import files

files.download("/content/helmet.pt")
files.download(str(latest_train / "results.png"))
files.download(str(latest_train / "confusion_matrix.png"))
files.download(str(latest_train / "labels.jpg"))

# 4. Opción B — Uso del modelo ya entrenado

Esta es la opción recomendada para presentaciones, demostraciones en clase o pruebas rápidas.

En lugar de volver a entrenar el modelo, se carga directamente el archivo `helmet.pt`.

## 4.1 Cargar el archivo `helmet.pt`

En esta sección se sube manualmente el modelo ya entrenado desde la computadora local.

In [ ]:
from google.colab import files
uploaded = files.upload()

## 4.2 Carga del modelo entrenado

In [ ]:
model = YOLO("/content/helmet.pt")
print("Modelo cargado correctamente.")

## 4.3 Subir una imagen para detección

En esta sección se carga una imagen externa y se genera una predicción con bounding boxes y etiquetas.

In [ ]:
uploaded = files.upload()
image_name = list(uploaded.keys())[0]

print("Imagen cargada:", image_name)

In [ ]:
model.predict(
    source=f"/content/{image_name}",
    save=True,
    conf=0.25
)

In [ ]:
from pathlib import Path
from IPython.display import Image, display

latest_predict_dir = sorted(Path("/content/runs/detect").glob("predict*"))[-1]
output_image = list(latest_predict_dir.glob("*"))[0]


## 4.4 Visualización del resultado

In [ ]:
display(Image(filename=str(output_image)))

# 5. Conclusión

Este notebook permite trabajar en dos modalidades:

- reentrenar el modelo desde cero a partir del dataset Hard Hat Workers
- utilizar directamente el modelo previamente entrenado (`helmet.pt`) para realizar inferencias rápidas

La segunda modalidad es especialmente útil para demostraciones en clase, ya que evita repetir el proceso de entrenamiento y permite obtener resultados en pocos segundos.